###Ingest CBS parquet file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe write api

In [0]:
dbutils.widgets.text("p_batch_date","")
v_batch_date = dbutils.widgets.get("p_batch_date")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{raw_path}/{v_batch_date}/cbs/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.cbs"

In [0]:
from pyspark.sql.types import *

In [0]:
cbs_schema = StructType([
  StructField('TransactionID', StringType(), True),
  StructField('BillerID', StringType(), True),
  StructField('CustomerID', StringType(), True),
  StructField('CreditAmount', DoubleType(), True),
  StructField('CreditStatus', StringType(), True),
  StructField('BankReferenceNo', StringType(), True),
  StructField('CreditTimestamp', DateType(), True)
])

In [0]:
cbs_df = spark.read.format('parquet') \
                .option('mode', 'FAILFAST') \
                .load(source_file)

In [0]:
cbs_audit = add_ingestion_metadata(cbs_df)

In [0]:
cbs_final= cbs_audit.withColumn("batch_date", F.lit(v_batch_date))

In [0]:
cbs_final.write.mode('overwrite').partitionBy('batch_date').option('replaceWhere',f"batch_date = '{v_batch_date}'").saveAsTable(table_name)

In [0]:
%sql
SELECT * FROM payment_app.bronze.cbs

TxnID,AccountNumber,TxnAmount,CBSStatus,RRN,TransactionDate,ingestion_timestamp,source_file,batch_date
TXN1006,100006,2300.0,Success,RRN100006,2026-08-07 10:05:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_002.parquet,2026-08-13
TXN1007,100007,1750.0,Success,RRN100007,2026-08-07 10:20:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_002.parquet,2026-08-13
TXN1008,100008,699.0,Success,RRN100008,2026-08-07 10:35:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_002.parquet,2026-08-13
TXN1009,100009,650.0,Pending,null,2026-08-07 10:45:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_002.parquet,2026-08-13
TXN1010,100010,399.0,Success,RRN100010,2026-08-07 11:00:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_002.parquet,2026-08-13
TXN1001,100001,1500.0,Success,RRN100001,2026-08-07 09:05:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_001.parquet,2026-08-13
TXN1002,100002,800.0,Success,RRN100002,2026-08-07 09:15:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_001.parquet,2026-08-13
TXN1003,100003,999.0,Pending,null,2026-08-07 09:30:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_001.parquet,2026-08-13
TXN1004,100004,1200.0,Failed,null,2026-08-07 09:40:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_001.parquet,2026-08-13
TXN1005,100005,450.0,Success,RRN100005,2026-08-07 09:50:00,2026-08-22T13:15:02.974Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/cbs/CBS_20260807_001.parquet,2026-08-13
